# 02 — Analysis

Persist the paired table using the project's columnar container
(`<table>/<column>.zst.partNN` inside an **uncompressed** tar), verify the
round-trip, and confirm the FAT32 constraint holds.

In [ ]:
import json
import shutil
import sys
import tarfile
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

from msys2_dataset.packages import pair_packages, parse_ls_listing_file
from msys2_dataset.store import FAT32_MAX_BYTES, read_table, write_table

FAT32_MAX_BYTES, FAT32_MAX_BYTES / 1024**3

### Build the table

In [ ]:
pairs = pair_packages(
    parse_ls_listing_file("data/msys2-pkgs.tsv"),
    parse_ls_listing_file("data/mingw-pkgs.tsv"),
)

table = {
    "name": [p.name for p in pairs],
    "msys2_name": [p.msys2.name if p.msys2 else None for p in pairs],
    "mingw_name": [p.mingw.name if p.mingw else None for p in pairs],
    "paired": [p.paired for p in pairs],
}
{column: len(values) for column, values in table.items()}

### Write it as a columnar zstd archive

In [ ]:
# Build into a scratch directory so running the notebook does not dirty the
# tracked tree; the real pipeline writes to data/processed/.
out = Path(".notebook-tmp/processed")
out.mkdir(parents=True, exist_ok=True)
archive = out / "packages.tar"
write_table(archive, "packages", table, max_part_bytes=1024 * 1024)
archive.stat().st_size

### Inspect the produced layout

In [ ]:
import tarfile

with tarfile.open(archive, mode="r:") as tar:  # "r:" rejects compression
    names = tar.getnames()
print(f"{len(names)} members")
names[:8]

### Verify the round-trip and the size cap

In [ ]:
import json

# Extract into a separate directory; deleting the parent would remove the
# archive we are reading from.
extract_dir = Path(".notebook-tmp/extracted")
shutil.rmtree(extract_dir, ignore_errors=True)
restored = read_table(archive, extract_dir)
assert restored == table, "round-trip mismatch"
print("round-trip OK")

with tarfile.open(archive, mode="r:") as tar:
    oversize = [m.name for m in tar.getmembers() if m.size > FAT32_MAX_BYTES]
assert not oversize, oversize
print("every member is within the FAT32 limit")

### SQLite alternative

For shards that are rewritten repeatedly, the SQLite path is faster than
re-serialising a whole column. SQLite columns are scalar, so `aliases` (a
list) is stored as a JSON string.

In [ ]:
from msys2_dataset.sqlite_store import load_rows, write_rows

db = out / "packages.sqlite"
records = [
    {
        "name": p.name,
        "msys2_name": p.msys2.name if p.msys2 else None,
        "mingw_name": p.mingw.name if p.mingw else None,
        "paired": int(p.paired),
        "aliases": json.dumps(list(p.aliases)),
    }
    for p in pairs
    if p.paired
]
write_rows(db, "paired", records, if_exists="replace")
rows = load_rows(db, "paired")
print(f"{len(rows)} paired rows in {db}")
rows[0]